        # 🧮 L03　運算子與 print 格式化
        **Python 冒險之旅 2026**　｜　Day 1（08/29 六）🏝️ 起始之島　｜　關卡　｜　🏅 100 XP

        📖 對應教科書：第 2 章 2.4、2.6


        ### 🎯 這一關你會學到
        - 使用算術、複合指定、關係與邏輯運算子
- 掌握運算子優先順序
- 用 % 格式字串與 f-string 控制輸出

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "L03"
_SALT = "python-quest-2026-datama"
_TASKS = ["3-1", "3-2", "3-3", "3-4", "3-5", "3-6"]
_XP_EACH = 16
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_3_1(run):
    out, ns = run()
    nums = 數字們(out)
    need = [22, 12, 85, 3.4, 3, 2, 1419857]
    missing = [n for n in need if n not in nums]
    return (not missing, f"少了這些結果：{missing}")
任務定義("3-1", _check_3_1, 提示="x / y 是 3.4，x // y 是 3，x % y 是 2，x ** y 是 1419857。")

def _check_3_2(run):
    out, ns = run()
    return ((ns.get("h"), ns.get("m"), ns.get("s")) == (1, 2, 5),
            f"h, m, s 應該是 1, 2, 5，現在是 {ns.get('h')}, {ns.get('m')}, {ns.get('s')}。")
任務定義("3-2", _check_3_2, 提示="h = total // 3600；剩下的秒數 total % 3600 再 // 60 得到分；s = total % 60。")

def _check_3_3(run):
    out, ns = run()
    src = run.src
    ops = sum(op in src for op in ["-=", "*=", "//="])
    if ops < 3: return (False, "要用到 -=、*=、//= 三種複合指定運算子。")
    return (ns.get("money") == 866, f"最後 money 應該是 866，現在是 {ns.get('money')}。")
任務定義("3-3", _check_3_3, 提示="money -= 200、money *= 2、money //= 3。")

def _check_3_4(run):
    out, ns = run()
    src = run.src
    if " or " not in src or " and " not in src:
        return (False, "要用到 or 與 and 運算子。")
    if ns.get("adult") is not True: return (False, "age=20 時 adult 應該是 True。")
    if ns.get("discount") is not False: return (False, "age=20 時 discount 應該是 False。")
    if ns.get("vip") is not True: return (False, "member=True 且成年，vip 應該是 True。")
    return True
任務定義("3-4", _check_3_4, 提示="adult = age >= 18；discount = age < 12 or age >= 65；vip = member and age >= 18。")

def _check_3_5(run):
    out, ns = run()
    p = ns.get("預測", {})
    actual = {"r1": 2.0, "r2": 512, "r3": -9, "r4": 27}
    wrong = [k for k in actual if p.get(k) != actual[k]]
    return (not wrong, f"預測錯的有：{wrong}。想想 ** 是由右往左、-3**2 先算次方。")
任務定義("3-5", _check_3_5, 提示="1 + 4*3/2%5 → 先 4*3=12 → /2=6.0 → %5=1.0 → 1+1.0=2.0；2**3**2 = 2**9。")

def _check_3_6(run):
    out, ns = run()
    if abs(float(ns.get("f", 0)) - 262.4) > 1e-6: return (False, "f 應該是 262.4。")
    if "0262.400" not in out: return (False, "華氏要顯示成 0262.400（總寬 8、小數 3 位、補 0）。")
    return (" 128" in out, "攝氏要佔 4 格，所以 128 前面會有一個空格。")
任務定義("3-6", _check_3_6, 提示="{c:4d} 佔 4 格；{f:08.3f} 總寬 8（含小數點）、小數 3 位、補 0。")


## 🧮 3-1　算術運算子
| 運算子 | 意義 | 例子（x=17, y=5） |
|---|---|---|
| `+ - *` | 加減乘 | `17 + 5 → 22` |
| `/` | 除（結果一定是浮點數） | `17 / 5 → 3.4` |
| `//` | **整數除法**（商） | `17 // 5 → 3` |
| `%` | **餘數** | `17 % 5 → 2` |
| `**` | 次方 | `2 ** 10 → 1024` |

`//` 和 `%` 是資料處理的好朋友：換算時分秒、判斷奇偶、分組……都靠它們。

In [ ]:
r = 6.4
PI = 3.14159
print("圓的半徑：", r)                 # 課本 ex02/arithmetic.py
print("圓面積：", PI * r ** 2)
print("圓周長：", PI * r * 2)
print("球的體積：", PI * r ** 3 * 4 / 3)
print(17 / 5, 17 // 5, 17 % 5)

## 3-2　複合指定運算子
`x += 1` 就是 `x = x + 1` 的縮寫，同理有 `-=`、`*=`、`/=`、`//=`、`%=`、`**=`。

In [ ]:
money = 1000
money += 500    # money = money + 500 → 1500
money -= 200    # 1300
money *= 2      # 2600
money //= 3     # 866
print(money)

## 3-3　關係運算子與邏輯運算子
- 關係運算子 `>  <  >=  <=  ==  !=` 的結果是 **布林值** `True / False`。注意 `==` 是「等於」，`=` 是「指定」！
- 邏輯運算子：`and`（而且）、`or`（或者）、`not`（不是）。
- `in` 可以判斷「有沒有包含」：`'P' in 'Python'` → `True`。

In [ ]:
a, b = 2, 3
print(a < b, a >= b, a == b, a != b)          # 課本 ex02/relational.py
print((1 < 2) and ('A' == 'a'))                # False：兩個都要成立才是 True
print((-1 < 0) or (-1 > 100))                  # True：一個成立就 True
print(not ('A' != 'a'))                        # False
print('P' in 'Python', 'x' not in 'Python')    # True True

## 3-4　運算子的優先順序（課本 2.4.9）
由高到低：`**` → 正負號 `+x -x` → `* / // %` → `+ -` → 關係運算子 → `not` → `and` → `or`。
**不確定就加括號**，既安全又好讀。

In [ ]:
print(1 + 4 * 3 / 2 % 5)   # 先 4*3=12 → 12/2=6.0 → 6.0%5=1.0 → 1+1.0 = 2.0
print(2 ** 3 ** 2)         # ** 由右往左：3**2=9 → 2**9 = 512
print(-3 ** 2)             # 先算 3**2 再加負號 → -9
print((-3) ** 2)           # 9

## 3-5　`print()` 的格式化輸出（課本 2.6）
兩種常用寫法，**f-string 是現代 Python 的首選**：

| 寫法 | 例子 | 結果 |
|---|---|---|
| `%` 格式字串 | `'%s 今年 %d 歲' % ('小明', 20)` | 小明 今年 20 歲 |
| `%` 小數 | `'%.2f' % 3.14159` | 3.14 |
| f-string | `f'{name} 今年 {age} 歲'` | 小明 今年 20 歲 |
| f-string 小數 | `f'{3.14159:.2f}'` | 3.14 |
| f-string 寬度／補零 | `f'{42:5d}'`、`f'{42:05d}'` | `   42`、`00042` |
| f-string 千分位 | `f'{1234567:,}'` | 1,234,567 |

In [ ]:
wt, price = 30, 20.5
print('%s%d斤,共%.1f元' % ('香蕉', wt, wt * price))     # 課本 ex02/print2.py
print('%8.2f|' % -12.3456)                              # 總寬 8、小數 2 位
print('%08d' % 12345)                                   # 補 0 到 8 位
name, bmi = "小美", 21.4567
print(f"{name} 的 BMI 是 {bmi:.1f}")
print(f"|{42:5d}|{42:05d}|{1234567:,}|{0.256:.1%}|")
print('台北', '台中', '台南', sep=',')     # sep：分隔字元
print('價目表：', end='')                  # end：結尾不換行
print('陽春麵', 30, '元')

### 🎯 任務 3-1　七種運算一次看

`x = 17`、`y = 5`。請依序印出 `x+y`、`x-y`、`x*y`、`x/y`、`x//y`、`x%y`、`x**y` 的結果（7 個值，可以用一個 print 用逗號隔開，也可以分 7 行）。

In [ ]:
# 🎯 任務 3-1　七種運算一次看（請保留這一行）
x, y = 17, 5
print(x + y, x - y)
# 繼續印出其餘五種運算

In [ ]:
檢查("3-1")   # ◀ 執行這一格，看看任務 3-1 有沒有過關

### 🎯 任務 3-2　秒數換算器

`total = 3725` 秒。請用 `//` 和 `%` 算出 `h`（小時）、`m`（分）、`s`（秒），並印出 `1 小時 2 分 5 秒`。

In [ ]:
# 🎯 任務 3-2　秒數換算器（請保留這一行）
total = 3725
h = ???
m = ???
s = ???
print(h, "小時", m, "分", s, "秒")

In [ ]:
檢查("3-2")   # ◀ 執行這一格，看看任務 3-2 有沒有過關

### 🎯 任務 3-3　金幣帳本

用**複合指定運算子**完成下面的帳本：初始 1000 金幣 → 賺 500 → 花 200 → 翻倍 → 平分給 3 人（整數除法）→ 最後印出剩餘金幣。

In [ ]:
# 🎯 任務 3-3　金幣帳本（請保留這一行）
money = 1000
money += 500
# 花掉 200
# 翻倍
# 平分給 3 人（整數除法）
print(money)

In [ ]:
檢查("3-3")   # ◀ 執行這一格，看看任務 3-3 有沒有過關

### 🎯 任務 3-4　入場條件

樂園規則：`adult`＝年齡滿 18；`discount`＝未滿 12 **或** 65 歲以上；`vip`＝是會員**而且**成年。請用關係與邏輯運算子寫出三個**運算式**（不要直接寫 True/False）。

In [ ]:
# 🎯 任務 3-4　入場條件（請保留這一行）
age = 20
member = True
adult = ???
discount = ???
vip = ???
print(adult, discount, vip)

In [ ]:
檢查("3-4")   # ◀ 執行這一格，看看任務 3-4 有沒有過關

### 🎯 任務 3-5　先猜再驗證：優先順序

**先不要執行**，把你對四個運算式結果的預測填進 `預測` 字典，再執行程式比對。四個都猜對才過關（猜錯很正常，想想為什麼！）。

In [ ]:
# 🎯 任務 3-5　先猜再驗證：優先順序（請保留這一行）
預測 = {"r1": 0, "r2": 0, "r3": 0, "r4": 0}   # ← 先填你的預測（r1 是浮點數）
r1 = 1 + 4 * 3 / 2 % 5
r2 = 2 ** 3 ** 2
r3 = -3 ** 2
r4 = (1 + 2) * 3 ** 2
print("實際結果：", r1, r2, r3, r4)
print("你的預測：", 預測)

In [ ]:
檢查("3-5")   # ◀ 執行這一格，看看任務 3-5 有沒有過關

### 🎯 任務 3-6　溫度報表

攝氏 128 度換算華氏（公式：華氏 = 攝氏 × 9 / 5 + 32）。請用 **f-string** 印出一行：攝氏佔 4 格、華氏整數 4 位小數 3 位、不足補 0：

`攝氏  128度 = 華氏0262.400度`

In [ ]:
# 🎯 任務 3-6　溫度報表（請保留這一行）
c = 128
f = ???
print(f"攝氏{c:???}度 = 華氏{f:???}度")

In [ ]:
檢查("3-6")   # ◀ 執行這一格，看看任務 3-6 有沒有過關

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：⚔️ B1 Boss 戰：旅費精算師** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/B1_boss_travel_budget.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/